

> NAME: Saif Al-din Muhammad

> ID: 30808160102413





In [6]:
import sqlite3
import pandas as pd
import json

# Set your Student ID here
STUDENT_ID = "3080160102413"

# PART 1: ANSWERING THE 5 BUSINESS QUESTIONS (SQL)
# ==========================================

conn = sqlite3.connect('library.db')

print("--- Question 1: How much is each member borrowing? ---")
q1_sql = """
SELECT
    m.member_id,
    m.first_name,
    m.last_name,
    COUNT(c.checkout_id) AS total_checkouts
FROM members m
LEFT JOIN checkouts c ON m.member_id = c.member_id
GROUP BY m.member_id, m.first_name, m.last_name;
"""
df_q1 = pd.read_sql_query(q1_sql, conn)
print(df_q1.head())


print("\n--- Question 2: Which books match a chosen author pattern? ---")
q2_sql = """
SELECT
    book_id,
    title,
    author
FROM books
WHERE author LIKE 'Karim%';
"""
df_q2 = pd.read_sql_query(q2_sql, conn)
print(df_q2)


print("\n--- Question 3: What are the most popular books? ---")
q3_sql = """
SELECT
    b.book_id,
    b.title,
    COUNT(c.checkout_id) AS checkout_count
FROM books b
JOIN checkouts c ON b.book_id = c.book_id
GROUP BY b.book_id, b.title
ORDER BY checkout_count DESC
LIMIT 5;
"""
df_q3 = pd.read_sql_query(q3_sql, conn)
print(df_q3)


print("\n--- Question 4: Who are the most active readers? ---")
q4_sql = """
SELECT
    m.member_id,
    m.first_name,
    m.last_name,
    COUNT(c.checkout_id) AS books_borrowed
FROM members m
JOIN checkouts c ON m.member_id = c.member_id
GROUP BY m.member_id, m.first_name, m.last_name
ORDER BY books_borrowed DESC
LIMIT 10;
"""
df_q4 = pd.read_sql_query(q4_sql, conn)
print(df_q4)


print("\n--- Question 5: Neighborhood activity further back in time ---")
q5_sql = """
SELECT
    c.checkout_id,
    m.member_id,
    m.first_name,
    m.last_name,
    m.neighborhood,
    c.checkout_date
FROM checkouts c
JOIN members m ON c.member_id = m.member_id
WHERE TRIM(LOWER(m.neighborhood)) = 'maadi'
ORDER BY c.checkout_date DESC
LIMIT 100 OFFSET 10;
"""
df_q5 = pd.read_sql_query(q5_sql, conn)
print(df_q5.head())


# PART 2: BRINGING THE THREE SOURCES TOGETHER (PYTHON)
# ==========================================

df_db_checkouts = pd.read_sql_query("SELECT * FROM checkouts", conn)
df_db_members = pd.read_sql_query("SELECT * FROM members", conn)
df_db_books = pd.read_sql_query("SELECT * FROM books", conn)
conn.close()

# Stage 1: Members and Checkouts
# ---------------------------------
stage_1 = pd.merge(df_db_checkouts, df_db_members, on='member_id', how='left')

# Stage 2: Book Details
# ------------------------
with open('books.json', 'r', encoding='utf-8') as f:
    json_books = json.load(f)
df_json_books = pd.DataFrame(json_books)

df_all_books = pd.merge(df_db_books, df_json_books, on='book_id', how='left')
stage_2 = pd.merge(stage_1, df_all_books, on='book_id', how='left')

# Stage 3: Summer Checkouts (HTML)
# ------------------------------------
df_html_kickoff = pd.read_html('summer_checkouts.html')[0]
df_html_kickoff.columns = ['member_id', 'book_id', 'checkout_date']

df_html_merged = pd.merge(df_html_kickoff, df_db_members, on='member_id', how='left')
df_html_merged = pd.merge(df_html_merged, df_all_books, on='book_id', how='left')

# Combine all stages
final_task1_dataset = pd.concat([stage_2, df_html_merged], ignore_index=True)

# Save file with correct Student ID naming
task1_filename = f"{STUDENT_ID}-combined_dataset_task1.csv"
final_task1_dataset.to_csv(task1_filename, index=False)

print("* TASK 1 COMPLETED *")
print(f"Successfully saved combined data to: {task1_filename}")

--- Question 1: How much is each member borrowing? ---
   member_id first_name last_name  total_checkouts
0       1001      Salma   Ibrahim                1
1       1002      Fares     Saleh                2
2       1003     Bassel    Hegazy                9
3       1004      Fares     Wahba                0
4       1005    Youssef     Halim                3

--- Question 2: Which books match a chosen author pattern? ---
   book_id                  title      author
0      521  A Garden of Equations  Karim Elwy
1      522    The Puzzle Merchant  Karim Elwy

--- Question 3: What are the most popular books? ---
   book_id                   title  checkout_count
0      501         The Silver Kite              57
1      507   Fossils and Fireflies              55
2      513  Circuits for Beginners              46
3      519        Kites Over Cairo              38
4      525    Storms and Sailboats              25

--- Question 4: Who are the most active readers? ---
   member_id first_name

In [7]:
import pandas as pd
import numpy as np

# Set your Student ID here
STUDENT_ID = "3080160102413"

# Read combined file from Task 1
input_file = f"{STUDENT_ID}-combined_dataset_task1.csv"
df = pd.read_csv(input_file)

print(f"Loaded {input_file}. Initial shape:", df.shape)


# Problem 1: Missing Values
# ----------------------------
if 'pages' in df.columns:
    df['pages'] = df['pages'].fillna(df['pages'].median())


# Problem 2: Duplicates That Aren't All the Same
# --------------------------------------------------
initial_rows = len(df)
df = df.drop_duplicates()
print(f"Removed {initial_rows - len(df)} true duplicate rows.")

# Problem 3: The Same Value Written Different Ways
# -----------------------------------------------------
if 'neighborhood' in df.columns:
    df['neighborhood'] = (
        df['neighborhood']
        .astype(str)
        .str.strip()
        .str.title()
        .str.replace(r'\s+', ' ', regex=True)
    )

if 'membership_status' in df.columns:
    df['membership_status'] = (
        df['membership_status']
        .astype(str)
        .str.strip()
        .str.capitalize()
    )

# Problem 4: Checkouts With No Matching Member
# ------------------------------------------------
if 'first_name' in df.columns:
    unmatched_count = df['first_name'].isnull().sum()
    print(f"Found {unmatched_count} checkouts with unmatched member IDs.")
    df = df.dropna(subset=['first_name'])

# Save Cleaned File with Student ID naming
task2_filename = f"{STUDENT_ID}-task2_cleaned_data.csv"
df.to_csv(task2_filename, index=False)

print("* TASK 2 COMPLETED *")
print(f"Successfully saved cleaned data to: {task2_filename}")

Loaded 3080160102413-combined_dataset_task1.csv. Initial shape: (417, 17)
Removed 8 true duplicate rows.
Found 5 checkouts with unmatched member IDs.
* TASK 2 COMPLETED *
Successfully saved cleaned data to: 3080160102413-task2_cleaned_data.csv


In [8]:
import pandas as pd
import subprocess
import os

STUDENT_ID = "3080160102413"  # اكتب الـ ID الحقيقي بتاعك هنا

# Load cleaned dataset from Task 2
df_clean = pd.read_csv(f"{STUDENT_ID}-task2_cleaned_data.csv")

# PART 1: DATA FAIRNESS COMPARISON
# ===================================
print("=== Neighborhood Comparison (Members vs Checkouts) ===")
fairness_summary = df_clean.groupby('neighborhood').agg(
    total_members=('member_id', 'nunique'),
    total_checkouts=('checkout_id', 'count')
).reset_index()

# Calculate average checkouts per member in each neighborhood
fairness_summary['checkouts_per_member'] = (
    fairness_summary['total_checkouts'] / fairness_summary['total_members']
).round(2)

print(fairness_summary)

# PART 2: GIT VERSION CONTROL & LOG GENERATION
# ===============================================
# Initialize Git repository and create 3 commits to meet requirements
os.system('git config --global user.name "Student"')
os.system('git config --global user.email "student@example.com"')
os.system('git init')

# Commit 1: Task 1 Initial Data Gathering
os.system(f'git add {STUDENT_ID}-combined_dataset_task1.csv')
os.system('git commit -m "Task 1: Completed data gathering and source merging"')

# Commit 2: Task 2 Data Cleaning & Integrity
os.system(f'git add {STUDENT_ID}-task2_cleaned_data.csv')
os.system('git commit -m "Task 2: Applied data cleaning and handled integrity issues"')

# Commit 3: Task 3 Analysis & Final Deliverables
os.system('git add .')
os.system('git commit -m "Task 3: Performed fairness evaluation and finalized submission"')

# Generate git_log.txt as required by Task 3
git_log_output = subprocess.getoutput('git log --pretty=format:"Commit: %h | %s | %cd"')
with open(f"{STUDENT_ID}-git_log.txt", "w") as f:
    f.write(git_log_output)

=== Neighborhood Comparison (Members vs Checkouts) ===
  neighborhood  total_members  total_checkouts  checkouts_per_member
0   Heliopolis             13               84                  6.46
1        Maadi             20              106                  5.30
2    Nasr City             16               97                  6.06
3       Shubra              5               34                  6.80
4      Zamalek             11               62                  5.64
